### Importar Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import os
import openpyxl

#### Limpeza da base de dados (CHIKON)

In [ ]:
# Carregar o arquivo Excel
file_path = "chikon_[sigla_estado].xlsx"
df_raw_chik = pd.read_excel(file_path)

# Remover colunas desnecessárias
colunas_para_remover = [
    'SOUNDEX', 'ID_DISTRIT', 'ID_GEO1', 'ID_GEO2', 'DIABETES', 'HEMATOLOG', 'HEPATOPAT', 'RENAL', 'HIPERTENSA',
    'ACIDO_PEPT', 'AUTO_IMUNE', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 
    'HOSPITALIZ', 'DT_INTERNA', 'UF', 'MUNICIPIO', 'HOSPITAL', 'DDD_HOSP', 'TEL_HOSP', 'TPAUTOCTO', 'COUFINF', 
    'COPAISINF', 'COMUNINF', 'CODISINF', 'CO_BAINF', 'NOBAIINF', 'DOENCA_TRA', 'CLINC_CHIK', 'ALRM_HIPOT', 
    'ALRM_PLAQ', 'ALRM_VOM', 'ALRM_SANG', 'ALRM_HEMAT', 'ALRM_ABDOM', 'ALRM_LETAR', 'ALRM_HEPAT', 'ALRM_LIQ', 
    'DT_ALRM', 'GRAV_PULSO', 'GRAV_CONV', 'GRAV_ENCH', 'GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 
    'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO', 
    'DT_GRAV', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 
    'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLICA', 'NU_LOTE_I', 'NDUPLIC_N', 'DT_TRANSUS', 
    'DT_TRANSDM', 'DT_TRANSSM', 'DT_TRANSRM', 'DT_TRANSRS', 'DT_TRANSSE', 'NU_LOTE_V', 'NU_LOTE_H', 'CS_FLXRET', 
    'FLXRECEBI', 'IDENT_MICR', 'MIGRADO_W', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE', 'ID_PAIS', 'ID_OCUPA_N', 
    'DT_SORO', 'DT_NS1', 'DT_VIRAL', 'SOROTIPO', 'RESUL_VI_N', 'HISTOPA_N', 'IMUNOH_N', 'TP_SISTEMA', 'TP_NOT', 
    'CS_ESCOL_N', 'CS_GESTANT', 'DS_OBS', 'DT_ALARM', 'NM_LOGRADO', 'NU_LOTE_N', 'ID_LOGRADO', 'NM_REFEREN', 'NU_DDD_TEL', 'NU_TELEFON',
    'IN_VINCULA', 'FONETICA_N']

colunas_existentes = [col for col in colunas_para_remover if col in df_raw_chik.columns]
df_raw_chik = df_raw_chik.drop(columns=colunas_existentes)

# Criação da coluna SE
if 'SEM_PRI' in df_raw_chik.columns:
    posicao_sem_pri = df_raw_chik.columns.get_loc('SEM_PRI')    
    valores_se = df_raw_chik['SEM_PRI'].astype(str).str[-2:]
    df_raw_chik.insert(loc=posicao_sem_pri + 1, column='SE', value=valores_se)

df_raw_chik.to_excel("chikon_notificados.xlsx", index=False)

print("Arquivo gerado com sucesso!")

Arquivo gerado com sucesso!


In [ ]:
# Contar ocorrências de casos notificados
df_raw_chik['ID_AGRAVO'].value_counts().reset_index(name='Total')

In [ ]:
# Total descartados
df_raw_chik['CLASSI_FIN'].value_counts().reset_index(name='Total')

### Descartados - Clínico Epidemiológico

In [ ]:
# 1. Carregar o arquivo Excel
file_path = 'chikon_notificados.xlsx'
df_clepi = pd.read_excel(file_path)

# 2. Filtrar as linhas (com .copy() para evitar avisos)
df_clepi['CRITERIO'] = pd.to_numeric(df_clepi['CRITERIO'], errors='coerce')
df_filtered_1 = df_clepi[df_clepi['CRITERIO'] == 2].copy()
df_filtered_1 = df_filtered_1[df_filtered_1['CLASSI_FIN'] == 5.0].copy()
df_filtered_1 = df_filtered_1[df_filtered_1['ID_MN_RESI'] == 500370].copy()

# Limpeza de Dados (garantindo que tudo é número)
cols_para_limpar = ['FEBRE', 'ARTRALGIA', 'ARTRITE', 'RESUL_PCR_']
for col in cols_para_limpar:
    df_filtered_1[col] = pd.to_numeric(df_filtered_1[col], errors='coerce')

# NOVAS COLUNAS: Mapeamento e tradução dos sintomas, conforme a definição de caso da Febre do Chikungunya
map_sintomas = {1: 'sim', 2: 'não'}
df_filtered_1['CHIK_FEBRE'] = df_filtered_1['FEBRE'].map(map_sintomas)
df_filtered_1['CHIK_ARTRALGIA'] = df_filtered_1['ARTRALGIA'].map(map_sintomas)
df_filtered_1['CHIK_ARTRITE'] = df_filtered_1['ARTRITE'].map(map_sintomas)

# 3. Lógica da regra de avaliação:

# Avaliar se o caso apresenta febre
cond_febre = df_filtered_1['FEBRE'] == 1

# Avaliar se o caso apresenta Artralgia OU Artrite (pelo menos um dos dois)
cond_articular = (df_filtered_1['ARTRALGIA'] == 1) | (df_filtered_1['ARTRITE'] == 1)

# Avaliar a realização de exames (células vazias ou valor 4 - não realizado)
cond_exame = df_filtered_1['RESUL_PCR_'].isna() | (df_filtered_1['RESUL_PCR_'] == 4)

# Aplicar a classificação cruzando as 3 condições (todas precisam ser verdadeiras)
df_filtered_1['DESCARTE_CRITERIO'] = np.where(cond_febre & cond_articular & cond_exame, 'n_adequado', 'adequado')

# 4. Filtrar os casos classificados como 'sim'
casos_suspeitos = df_filtered_1[df_filtered_1['DESCARTE_CRITERIO'] == 'n_adequado']

# 5. Salvar o resultado
output_file_path = 'descartados_2026_clep.xlsx'     # Nome do arquivo de saída, conforme preferência
with pd.ExcelWriter(output_file_path) as writer:
    df_filtered_1.to_excel(writer, sheet_name='clinico_epidemiologico', index=False)

print("Arquivo gerado com sucesso!")

Arquivo gerado com sucesso!


### Descartados - Laboratorial

In [ ]:
# 1. Carregar o arquivo Excel
file_path = 'chikon_notificados.xlsx'
df_lab = pd.read_excel(file_path)

# 2. Filtrar as linhas (Corrigida a variável de df_clepi para df_lab)
df_lab['CRITERIO'] = pd.to_numeric(df_lab['CRITERIO'], errors='coerce')
df_filtered_1 = df_lab[df_lab['CRITERIO'] == 1].copy()
df_filtered_1 = df_filtered_1[df_filtered_1['CLASSI_FIN'] == 5.0].copy()
df_filtered_1 = df_filtered_1[df_filtered_1['ID_MN_RESI'] == 500370].copy()

# Limpeza de Dados (garantindo que tudo é número)
cols_para_limpar = ['FEBRE', 'ARTRALGIA', 'ARTRITE', 'RESUL_PCR_']
for col in cols_para_limpar:
    df_filtered_1[col] = pd.to_numeric(df_filtered_1[col], errors='coerce')

# NOVAS COLUNAS: Mapeamento e tradução dos sintomas, conforme a definição de caso da Febre do Chikungunya
map_sintomas = {1: 'sim', 2: 'não'}
df_filtered_1['CHIK_FEBRE'] = df_filtered_1['FEBRE'].map(map_sintomas)
df_filtered_1['CHIK_ARTRALGIA'] = df_filtered_1['ARTRALGIA'].map(map_sintomas)
df_filtered_1['CHIK_ARTRITE'] = df_filtered_1['ARTRITE'].map(map_sintomas)

# 3. CÁLCULO DE DATAS
df_filtered_1['DT_PCR'] = pd.to_datetime(df_filtered_1['DT_PCR'], errors='coerce')
df_filtered_1['DT_SIN_PRI'] = pd.to_datetime(df_filtered_1['DT_SIN_PRI'], errors='coerce')

# Calcular a diferença de dias entre a data da coleta e a data de início dos sintomas
df_filtered_1['DIF_DIAS'] = (df_filtered_1['DT_PCR'] - df_filtered_1['DT_SIN_PRI']).dt.days

# 4. LÓGICA DE DESCARTE

# 4.1 Condições Clínicas
cond_febre = df_filtered_1['FEBRE'] == 1
cond_articular = (df_filtered_1['ARTRALGIA'] == 1) | (df_filtered_1['ARTRITE'] == 1)
clinica_suspeita = cond_febre & cond_articular  # Febre + (Artralgia e/ou Artrite)

# 4.2 Condições Laboratoriais (Assumindo 1 = Detectável, 2 = Não Detectável, 4 = Não Realizado)
exame_nao_realizado = df_filtered_1['RESUL_PCR_'].isna() | (df_filtered_1['RESUL_PCR_'] == 4)
exame_negativo = df_filtered_1['RESUL_PCR_'] == 2
exame_positivo = df_filtered_1['RESUL_PCR_'] == 1

# 4.3 Condição de Tempo (Oportuno = 0 a 8 dias)
exame_oportuno = (df_filtered_1['DIF_DIAS'] >= 0) & (df_filtered_1['DIF_DIAS'] <= 8)
exame_fora_oportuno = df_filtered_1['DIF_DIAS'] > 8

# 5. DIAGNÓSTICO OPORTUNO (Preenchimento da coluna)
df_filtered_1['DIAG_OPORTUNO'] = 'não'
df_filtered_1.loc[exame_oportuno, 'DIAG_OPORTUNO'] = 'sim'
df_filtered_1.loc[df_filtered_1['DIF_DIAS'].isna(), 'DIAG_OPORTUNO'] = 'n_realizado'

# 6. Aplicação de Regras de Descarte

# REGRA N_ADEQUADO
# Tem clínica suspeita E (exame não foi feito OU exame foi negativo mas colhido fora do período oportuno)
descarte_inadequado = (clinica_suspeita & (exame_nao_realizado | (exame_negativo & exame_fora_oportuno))) | exame_positivo

# REGRA ADEQUADO
# NÃO tem clínica suspeita E exame é negativo colhido no tempo certo
regra_adequada = (~clinica_suspeita) & exame_negativo & exame_oportuno

# Coluna Output
# Se for da regra n_adequado, recebe 'n_adequado' (descartado sem critério). Caso contrário, 'adequado' (com critério).
df_filtered_1['DESCARTE_CRITERIO'] = np.where(descarte_inadequado, 'n_adequado', 'adequado')

# 6. FILTROS E CONTAGENS FINAIS
# Filtrar os casos classificados como 'n_adequado'
casos_suspeitos = df_filtered_1[df_filtered_1['DESCARTE_CRITERIO'] == 'n_adequado']

# 6. Salvar o resultado
output_file_path = 'descartados_2026_lab.xlsx'      # Nome do arquivo de saída, conforme preferência
with pd.ExcelWriter(output_file_path) as writer:
    df_filtered_1.to_excel(writer, sheet_name='laboratorial', index=False)

print("Arquivo gerado com sucesso!")

Arquivo gerado com sucesso!


#### Casos em investigação

In [ ]:
# 1. Carregar o arquivo Excel
file_path = 'chikon_notificados.xlsx'
df_lab = pd.read_excel(file_path)

# 2. Filtrar as linhas
df_lab['CRITERIO'] = pd.to_numeric(df_lab['CRITERIO'], errors='coerce')
df_filtered_1 = df_lab[df_lab['CRITERIO'] == 3].copy()
df_filtered_1 = df_filtered_1[df_filtered_1['CLASSI_FIN'].isnull()].copy()
df_filtered_1 = df_filtered_1[df_filtered_1['ID_MN_RESI'] == 500370].copy()

# Limpeza de Dados (garantindo que tudo é número)
cols_para_limpar = ['FEBRE', 'ARTRALGIA', 'ARTRITE', 'RESUL_PCR_']
for col in cols_para_limpar:
    df_filtered_1[col] = pd.to_numeric(df_filtered_1[col], errors='coerce')

# Tradução dos Sintomas
map_sintomas = {1: 'sim', 2: 'não'}
df_filtered_1['CHIK_FEBRE'] = df_filtered_1['FEBRE'].map(map_sintomas)
df_filtered_1['CHIK_ARTRALGIA'] = df_filtered_1['ARTRALGIA'].map(map_sintomas)
df_filtered_1['CHIK_ARTRITE'] = df_filtered_1['ARTRITE'].map(map_sintomas)

# 3. Lógica Ajustada da Regra:
cond_febre = df_filtered_1['FEBRE'] == 1
cond_articular = (df_filtered_1['ARTRALGIA'] == 1) | (df_filtered_1['ARTRITE'] == 1)
cond_exame = df_filtered_1['RESUL_PCR_'].isna() | (df_filtered_1['RESUL_PCR_'] == 4)

cenario_clinico = cond_febre & cond_articular & cond_exame

df_filtered_1['DESCARTE_CRITERIO'] = np.where(cenario_clinico, 'avaliar', 'descartável')

# 4. Filtrar os casos classificados como 'sim'
casos_suspeitos = df_filtered_1[df_filtered_1['DESCARTE_CRITERIO'] == 'avaliar']

# 8. Salvar o resultado
output_file_path = 'descartados_2026_invest.xlsx'     # Nome do arquivo de saída, conforme preferência
with pd.ExcelWriter(output_file_path) as writer:
    df_filtered_1.to_excel(writer, sheet_name='investigacao', index=False)
    
print("Arquivo gerado com sucesso!")

Arquivo gerado com sucesso!


#### Mapear casos por bairro

In [ ]:
file_path = "chikon_notificados.xlsx"
df_bairros = pd.read_excel(file_path)

# Filtrar municipio de residencia
df_bairros['ID_MN_RESI'] = pd.to_numeric(df_bairros['ID_MN_RESI'], errors='coerce')
df_bairros = df_bairros[df_bairros['ID_MN_RESI'] == 500370].copy()  # filtro apenas para Dourados

coluna_bairro = 'NM_BAIRRO'
coluna_status = 'CLASSI_FIN'

# 1. Criar tabela cruzada para contar os casos por bairro e status de classificação final
analise_bairros = pd.crosstab(
    index= df_bairros[coluna_bairro], 
    columns= df_bairros[coluna_status], 
    margins=True,
    margins_name='Total Notificados'
)

# 2. Ordenar o resultado para ver os bairros com mais notificações no topo
analise_bairros = analise_bairros.sort_values(by='Total Notificados', ascending=False)

# Visualizar o resultado na tela
print("\n--- ANÁLISE DE CASOS POR BAIRRO ---")
print(analise_bairros.head(20)) # Mostra os 20 principais bairros

# 3. Salvar essa análise em um arquivo Excel separado para você gerar gráficos depois
analise_bairros.to_excel("chik_analise_bairro.xlsx", sheet_name= "bairros")

print("Arquivo gerado com sucesso!")

### Concatenar arquivos e gerar planilha de análise

In [ ]:
# Carregar planilhas separadas
df_notificados = pd.read_excel('chikon_notificados.xlsx')
df_clepi = pd.read_excel('descartados_2026_clep.xlsx', sheet_name='clinico_epidemiologico')
df_lab = pd.read_excel('descartados_2026_lab.xlsx', sheet_name='laboratorial')
df_invest = pd.read_excel('descartados_2026_invest.xlsx', sheet_name='investigacao')

# Concatenar as planilhas
with pd.ExcelWriter('chik_descartados.xlsx') as writer:
    df_notificados.to_excel(writer, sheet_name='notificados', index=False)
    df_lab.to_excel(writer, sheet_name='laboratorial', index=False)
    df_clepi.to_excel(writer, sheet_name='clinico_epidemiologico', index=False)
    df_invest.to_excel(writer, sheet_name='investigacao', index=False)

print('Planilha gerada com sucesso!')

Planilha gerada com sucesso!


##### Algoritmo produzido por Dengue Brazil